# Discovery + Silver: `billing.payments`

Ultima tabla del dominio `billing`. Grano: un pago aplicado a una factura (puede haber varios pagos por factura).

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.billing__payments", engine)
df.shape

(80000, 8)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

payment_id              object
amount                  object
paid_at                 object
method                  object
invoice_id              object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,payment_id,amount,paid_at,method,invoice_id,_source_file,_ingested_at,_dag_run_id
0,PAY-00000001,87.44,2023-11-15,card,INV-00011046,billing/payments.csv,2026-07-21 15:02:09.492470,manual__2026-07-21T15:01:58.988673+00:00
1,PAY-00000002,22.74,2025-09-20,card,INV-00031283,billing/payments.csv,2026-07-21 15:02:09.492470,manual__2026-07-21T15:01:58.988673+00:00
2,PAY-00000003,55.13,2022-11-11,card,INV-00033793,billing/payments.csv,2026-07-21 15:02:09.492470,manual__2026-07-21T15:01:58.988673+00:00
3,PAY-00000004,42.48,2023-09-21,card,INV-00037256,billing/payments.csv,2026-07-21 15:02:09.492470,manual__2026-07-21T15:01:58.988673+00:00
4,PAY-00000005,359.2,2025-05-24,card,INV-00006722,billing/payments.csv,2026-07-21 15:02:09.492470,manual__2026-07-21T15:01:58.988673+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("payment_id duplicados:", df["payment_id"].duplicated().sum())

invoices = pd.read_sql("SELECT invoice_id FROM silver.billing__invoices", engine)
print("invoice_id huerfanos:", (~df["invoice_id"].isin(invoices["invoice_id"])).sum())

Nulos por columna:
payment_id      0
amount          0
paid_at         0
method          0
invoice_id      0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

payment_id duplicados: 0


invoice_id huerfanos: 0


## 3. `method` y `amount`

In [4]:
print("method:")
print(df["method"].value_counts())
print()
amount = pd.to_numeric(df["amount"], errors="coerce")
print("amount <= 0:", (amount <= 0).sum())
print(amount.describe())

method:
method
card             43990
bank_transfer    24054
paypal            7987
cash              3969
Name: count, dtype: int64

amount <= 0: 0
count    80000.000000
mean        81.171059
std        103.606248
min          1.100000
25%         25.000000
50%         49.750000
75%         97.882500
max       3987.090000
Name: amount, dtype: float64


## 4. Conclusion

Tabla limpia (sin nulos, sin duplicados, 0 FKs huerfanas, `amount` siempre positivo). Solo tipado y estandarizacion.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["payment_id", "invoice_id", "amount", "method", "paid_at"]].copy()

df_silver["amount"] = pd.to_numeric(df_silver["amount"], errors="raise")
df_silver["method"] = df_silver["method"].str.strip().str.lower()
df_silver["paid_at"] = pd.to_datetime(df_silver["paid_at"])

df_silver.head()

,payment_id,invoice_id,amount,method,paid_at
0,PAY-00000001,INV-00011046,87.44,card,2023-11-15
1,PAY-00000002,INV-00031283,22.74,card,2025-09-20
2,PAY-00000003,INV-00033793,55.13,card,2022-11-11
3,PAY-00000004,INV-00037256,42.48,card,2023-09-21
4,PAY-00000005,INV-00006722,359.20,card,2025-05-24


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["payment_id"].is_unique
assert df_silver["invoice_id"].isin(invoices["invoice_id"]).all()
assert df_silver["amount"].gt(0).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 80000 filas listas para silver


## 7. Escribir en `silver.billing__payments`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "billing.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.billing__payments CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "billing__payments",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
print("Escrito en silver.billing__payments")

OK: billing.sql ejecutado


Escrito en silver.billing__payments


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.billing__payments LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT payment_id) AS ids_unicos FROM silver.billing__payments", engine))
check

   filas  ids_unicos
0  80000       80000


,payment_id,invoice_id,amount,method,paid_at,_silver_loaded_at
0,PAY-00000001,INV-00011046,87.44,card,2023-11-15,2026-07-21 15:04:11.580135+00:00
1,PAY-00000002,INV-00031283,22.74,card,2025-09-20,2026-07-21 15:04:11.580135+00:00
2,PAY-00000003,INV-00033793,55.13,card,2022-11-11,2026-07-21 15:04:11.580135+00:00
3,PAY-00000004,INV-00037256,42.48,card,2023-09-21,2026-07-21 15:04:11.580135+00:00
4,PAY-00000005,INV-00006722,359.20,card,2025-05-24,2026-07-21 15:04:11.580135+00:00
